# VISCERA — broader anatomy wording, run all

Select **Runtime → Change runtime type → T4 GPU** (A100 also works), then **Run all**. Approve the Google Drive mount. No folder IDs, dataset paths, or model tokens need editing. The public Qwen base downloads automatically. Two training epochs use the complete training partition; checkpoints are stored on Drive every 100 optimizer steps. If Colab disconnects, reconnect to a GPU and run all again to resume exactly the same run.

The model learns bounded visual-authoring intent, including healthier, milder pathology, proportions, material adjectives, and preservation constraints. It does not infer a patient's diagnosis or simulate surgery. The dataset is synthetic authored supervision, with phrase-family-separated validation/audit and an independent agent-authored regression challenge. No clinician review is claimed.

The final cell downloads **viscera-wording-weights.zip** only if independent generation tests pass. VISCERA's running local studio watches Downloads for this archive and validates/activates it automatically. A failed gate saves a detailed report and keeps the existing model; training loss is never substituted for wording accuracy.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
from google.colab import drive, files
drive.mount('/content/drive')
DELIVERY = Path('/content/drive/MyDrive') / 'VISCERA Wording Training 2026-10-07'
ARCHIVE = DELIVERY / 'viscera-wording-training-v2.zip'
EXPECTED_SHA256 = '83a631a7564c5cc2a6a2c3f2b9196e2be9a0be03ed2028172b5a7451f2ea8ddb'
assert ARCHIVE.is_file(), f'Missing prepared dataset/source bundle: {ARCHIVE}'
with ARCHIVE.open('rb') as handle:
    assert hashlib.file_digest(handle, 'sha256').hexdigest() == EXPECTED_SHA256, 'Bundle checksum mismatch'
WORK = Path('/content/viscera-wording-v2')
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)), 'Duplicate ZIP entries'
    for item in archive.infolist():
        p = Path(item.filename)
        assert not p.is_absolute() and '..' not in p.parts and p.parts[0] == 'viscera-wording-training', 'Unsafe bundle path'
        assert (item.external_attr >> 16) & 0o170000 != 0o120000, 'Symlink is not allowed'
    archive.extractall(WORK)
ROOT = WORK / 'viscera-wording-training'
manifest = json.loads((ROOT / 'bundle_manifest.json').read_text())
for name, expected in manifest['artifact_hashes'].items():
    with (ROOT / name).open('rb') as handle:
        assert hashlib.file_digest(handle, 'sha256').hexdigest() == expected, f'Changed source or dataset: {name}'
DATA = ROOT / 'dataset'
RUN = DELIVERY / 'run-v2'
RUN.mkdir(exist_ok=True)
print('Verified dataset and frozen source:', manifest['dataset_release_sha256'])
print('Release conformance:', json.loads((ROOT / 'preflight.json').read_text()))
print('Persistent run:', RUN)

In [ ]:
"""Colab setup without a dependency on the runtime's optional ensurepip package."""

import subprocess
import sys
from pathlib import Path


def ensure_wording_environment(environment):
    environment = Path(environment).absolute()
    python = environment / "bin/python"

    def checked(command):
        result = subprocess.run(command, capture_output=True, text=True)
        if result.returncode:
            details = "\n".join(part.strip() for part in (result.stdout, result.stderr) if part)
            raise RuntimeError(
                f"Wording environment setup failed (exit {result.returncode}): "
                f"{command!r}\n{details}"
            )
        return result.stdout.strip()

    # Re-running this also repairs a partial venv left by a failed ensurepip step.
    # No --clear: installed packages and every Drive training checkpoint survive.
    checked([sys.executable, "-m", "venv", "--without-pip", str(environment)])
    checked([
        str(python), "-c",
        "import sys; from pathlib import Path; "
        "assert sys.prefix != sys.base_prefix, 'Expected an isolated environment'; "
        "assert Path(sys.prefix).resolve() == Path(sys.argv[1]).resolve(), "
        "'Unexpected environment prefix'",
        str(environment),
    ])
    probe = subprocess.run(
        [str(python), "-m", "pip", "--version"], capture_output=True, text=True
    )
    if probe.returncode:
        # pip's documented --python support can seed a pip-free environment.
        # This installs only into the target environment, not the Colab kernel.
        checked([
            sys.executable, "-m", "pip", "--python", str(python), "install",
            "--disable-pip-version-check", "--index-url", "https://pypi.org/simple",
            "pip==25.2",
        ])
    print("Wording environment ready:", python)
    print(checked([str(python), "-m", "pip", "--version"]))
    return python


# All training runs in an isolated Python environment; no kernel restart is needed.
ENV = Path('/content/viscera-wording-env')
PYTHON = ensure_wording_environment(ENV)
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check', 'torch==2.8.0', '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check', '-r', str(ROOT / 'requirements-wording-colab.txt')], check=True)
ENVIRONMENT = dict(os.environ, PYTHONPATH=str(ROOT / 'source/py') + os.pathsep + str(ROOT / 'source'), TOKENIZERS_PARALLELISM='false')
subprocess.run([str(PYTHON), '-c', "import torch; print(torch.__version__); assert torch.cuda.is_available(), 'Choose a GPU runtime before Run all'; print(torch.cuda.get_device_name(0)); a=torch.ones(16,device='cuda'); assert a.sum().item()==16"], check=True, env=ENVIRONMENT)
def wording(action, *arguments):
    subprocess.run([str(PYTHON), '-u', '-m', 'pipelines.train.viscera_train.atlas_wording', action, *map(str, arguments)], check=True, cwd=ROOT / 'source', env=ENVIRONMENT)

In [ ]:
# Actual tiny Qwen LoRA optimizer updates, loss decrease and exact checkpoint resume.
# These randomly initialized smoke weights are never exportable.
wording('smoke', '--data', DATA, '--out', ROOT / 'cpu-smoke')
smoke_report = json.loads((ROOT / 'cpu-smoke/smoke_report.json').read_text())
assert smoke_report['passed']
print(json.dumps(smoke_report, indent=2))

In [ ]:
# Public, immutable Qwen2.5-0.5B-Instruct base; complete train partition, two epochs.
# Completion-only cross entropy; rank-16 LoRA on attention and MLP projections.
# Dynamic padding, gradient accumulation/checkpointing, bf16 on A100/fp16 on T4.
arguments = ['--data', DATA, '--out', RUN, '--device', 'cuda', '--epochs', '2']
if (RUN / 'checkpoint.pt').exists():
    arguments.append('--resume')
wording('train', *arguments)
training_report = json.loads((RUN / 'training_report.json').read_text())
assert training_report['pretrained_base_loaded'] and not training_report['identity']['tiny_fixture']
print('Completed optimizer steps:', training_report['steps'])
print('Validation:', training_report['validation'])
print('Frozen-base comparison on identical validation prompts:', training_report['baseline_comparison'])

In [ ]:
# Only now open the held-out audit/challenge partitions. They never drive updates.
wording('evaluate', '--data', DATA, '--out', RUN, '--device', 'cuda')
evaluation = json.loads((RUN / 'evaluation.json').read_text())
print(json.dumps({'passed': evaluation['passed'], 'gates': evaluation['gates'], 'audit_operations_accuracy': evaluation['audit']['operations_accuracy'], 'challenge_operations_accuracy': evaluation['challenge']['operations_accuracy']}, indent=2))
if not evaluation['passed']:
    files.download(str(RUN / 'evaluation.json'))
    raise RuntimeError('Independent wording gates failed. Evaluation report saved/downloaded; no weights were promoted. Do not tune against this blind audit. Review validation errors and create a new untouched audit for a revised release.')

In [ ]:
OUTPUT = DELIVERY / 'viscera-wording-weights.zip'
wording('export', '--out', RUN, '--destination', OUTPUT)
print('Validated weights saved on Drive:', OUTPUT)
print('Keep the local VISCERA studio running and save this download in your normal Downloads folder.')
files.download(str(OUTPUT))